# 07 — Gaussian germanene ribbon with partial OBC

This calculation is **already evaluated at full settings**: 81 supercell momenta,
1201 energies on [-0.30,0.30] eV and 5 meV Gaussian broadening. The ribbon is
periodic along x and open across y, with two zigzag edges. Its length is
10.050 nm and width 10.212 nm (1500 atoms, 3000 spin orbitals).

The bubble has **height 1.5 nm, radius 2.5 nm and centre (5.025,1.5) nm**,
overlapping the lower edge. Radius means three Gaussian standard deviations.
The lower-edge projector selects atoms within 1 nm of that edge.
Both profiles use the same dimensions, height, radius, centre and spectral settings.

The model includes intrinsic SOC 100 meV and zero intrinsic Rashba. The
spectral functions are unfolded into the primitive ribbon Brillouin zone;
spatial LDOS is averaged over all sampled supercell momenta and summed over spins.
The Gaussian is periodized by summing images along the ribbon.

Saved full results are in `results/ribbons_partial_obc/07_gaussian_partial_obc/full/`.
The notebook names and outdated descriptions were corrected on 26 October 2026;
the saved numerical results were checked against the current geometry and settings.
The stored execution outputs originate from the existing full run.


In [ ]:
# Resolve the shared model and output root from this notebook or the project root.
from pathlib import Path
import os
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'model.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import json
import os
from dataclasses import asdict
from pathlib import Path
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.special import ndtr
import tbplas as tb
from model import ExportParams, _rect_cell

WIDTH_NM, LENGTH_NM = 10.0, 10.0
HEIGHT_NM, RADIUS_NM = 1.5, 2.5
EDGE_DEPTH_NM = 1.0
QUICK = os.getenv('HB_QUICK', '0') == '1'
NK = 17 if QUICK else 81  # unique, uniformly weighted points; no duplicated BZ endpoint
SPECTRAL_ENERGY_MIN_EV, SPECTRAL_ENERGY_MAX_EV = -0.30, 0.30
ENERGY = np.linspace(SPECTRAL_ENERGY_MIN_EV, SPECTRAL_ENERGY_MAX_EV,
                     451 if QUICK else 1201)
BROADENING_EV = 0.005  # Gaussian standard deviation
MAP_ENERGIES = np.array([-0.05, 0.0, 0.05])
GAP_WINDOW = np.array([-0.08, 0.08])  # strictly inside the flat bulk gap (~ +/-0.1 eV)
EIGEN_CUTOFF = max(abs(ENERGY).max(), abs(GAP_WINDOW).max(), abs(MAP_ENERGIES).max()) + 8*BROADENING_EV
INITIAL_EIGENPAIRS = 96
MAX_EIGENPAIRS = 1000
p = ExportParams(profile='gaussian', height=HEIGHT_NM, radius=RADIUS_NM, lambda_r2_mev=0.0)
nx = max(3, int(round(LENGTH_NM / p.a)))
ny = max(2, int(round((WIDTH_NM / p.bond + 1) / 3)))
L = nx*p.a
WIDTH = (3*ny-1)*p.bond
# Bubble centre in nm; define after L so a fresh kernel supports L/2.
BUBBLE_X_NM = L/2
BUBBLE_Y_NM = 1.5  # positive moves into the ribbon
assert np.isfinite(BUBBLE_X_NM) and np.isfinite(BUBBLE_Y_NM)
BUBBLE_X_NM = BUBBLE_X_NM % L  # equivalent centre in the periodic cell
q = np.linspace(-0.5, 0.5, NK, endpoint=False)  # q = k L / (2 pi)
k_nm = 2*np.pi*q/L
# Each supercell q has nx primitive momenta (q + m)/nx.
primitive_q_unsorted = (((q[:, None] + np.arange(nx)[None, :])/nx + 0.5) % 1 - 0.5).ravel()
unfold_order = np.argsort(primitive_q_unsorted)
primitive_q = primitive_q_unsorted[unfold_order]  # ka/(2 pi), full primitive BZ
primitive_k_nm = 2*np.pi*primitive_q/p.a
np.testing.assert_allclose(np.diff(primitive_q), 1/(NK*nx), atol=1e-12)
out = Path('results/ribbons_partial_obc/07_gaussian_partial_obc') / ('quick' if QUICK else 'full')
out.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'text.usetex': False})
assert 0 < EDGE_DEPTH_NM < WIDTH/2
assert p.lambda_r2_mev == 0, 'The spin-block solver below requires conserved spin.'
print(f'{nx} x {ny} rectangular cells; length {L:.3f} nm; atomic width {WIDTH:.3f} nm')
print(f'{4*nx*ny:,} physical atoms; {8*nx*ny:,} spin orbitals; {NK} k points')


## Geometry and Bloch Hamiltonian

Here $x$ is the long periodic direction and $y$ is the open direction (the native lattice coordinates, rather than the rotated display coordinates of 01–06). The rectangular-cell termination leaves two nearest neighbours on each outermost row, producing ordinary zigzag edges, not bearded edges. The notebook checks this explicitly.

The Gaussian is periodized by summing images along $x$; its centre is set by `BUBBLE_X_NM` and `BUBBLE_Y_NM` in the first code cell. Positive Y moves into the ribbon from the lower edge; X is wrapped into the periodic cell. The geometry marker and saved metadata use the same centre. The normal displacement of the two buckled sublattices and hopping law follow `model.py`. The same site ordering and edge projector are used in both cases. Bloch phases are applied only to bonds crossing the long supercell boundary, using $\exp(i k m L)$ for a bond to image $m$. This is a cell-periodic gauge; diagonal spectral weights are gauge invariant.


In [ ]:
def build_ribbon(height_nm):
    cell, basis, sub_a = _rect_cell(p)
    primitive = np.tile(np.arange(8), nx*ny)
    ix = np.repeat(np.arange(nx), ny*8)
    iy = np.tile(np.repeat(np.arange(ny), 8), nx)
    is_a = sub_a[primitive//2]
    spin = primitive % 2
    x = ix*p.a + basis[primitive//2, 0]
    y = iy*3*p.bond + basis[primitive//2, 1] - p.bond
    xy = np.column_stack((x[::2], y[::2]))
    sigma = RADIUS_NM/3
    z = np.zeros_like(x)
    hx, hy = np.zeros_like(x), np.zeros_like(x)
    # Include enough images for any selected Gaussian width; normalize peak to height_nm.
    images = int(np.ceil(8*sigma/L)) + 1
    norm = sum(np.exp(-(m*L)**2/(2*sigma**2)) for m in range(-images, images+1))
    for m in range(-images, images+1):
        dx = x-BUBBLE_X_NM-m*L
        dy = y-BUBBLE_Y_NM
        term = height_nm/norm*np.exp(-(dx*dx+dy*dy)/(2*sigma*sigma))
        z += term
        hx -= dx*term/sigma**2
        hy -= dy*term/sigma**2
    normal = np.column_stack((-hx, -hy, np.ones_like(x)))
    normal /= np.linalg.norm(normal, axis=1)[:, None]
    zb = np.where(is_a, p.buckle/2, -p.buckle/2)
    positions = np.column_stack((x, y, z)) + zb[:, None]*normal
    def deform(pos):
        assert pos.shape == positions.shape
        pos[:] = positions
    sc = tb.SuperCell(cell, dim=(nx, ny, 1), pbc=(True, False, False),
                      orb_pos_modifier=deform)
    sample = tb.Sample(sc)
    sample.init_orb_pos()
    sample.init_orb_eng()
    sample.init_hop()
    np.testing.assert_allclose(sample.orb_pos, positions, atol=1e-12)
    ii, jj = sample.hop_i, sample.hop_j
    nn = (is_a[ii] != is_a[jj]) & (spin[ii] == spin[jj]) & (abs(sample.hop_v.real) > 0.5)
    distance = np.linalg.norm(sample.dr[nn], axis=1)
    n2 = (sample.dr[nn, 2]/distance)**2
    cutoff = 1/(1+np.exp((distance-3*p.d0)/0.01))
    sample.hop_v[nn] = -(1-n2)*p.gamma0*np.exp(2*(p.d0-distance)/p.d0)*cutoff
    translation = sample.dr - (positions[jj]-positions[ii])
    image_x = np.rint(translation[:, 0]/L).astype(int)
    np.testing.assert_allclose(translation[:, 0], image_x*L, atol=1e-10)
    np.testing.assert_allclose(translation[:, 1:], 0, atol=1e-10)
    coordination = np.bincount(np.r_[ii[nn], jj[nn]], minlength=len(x))[::2]
    outer = np.isclose(xy[:, 1], 0) | np.isclose(xy[:, 1], WIDTH)
    assert np.all(coordination[outer] == 2), 'Termination is not ordinary zigzag.'
    assert np.all(coordination[~outer] == 3)
    edge = xy[:, 1] <= EDGE_DEPTH_NM
    # _rect_cell stores spin-flip Rashba entries even when their amplitudes are zero.
    # Spin conservation constrains hopping amplitudes, not the stored index pairs.
    spin_flip = spin[ii] != spin[jj]
    assert np.all(sample.hop_v[spin_flip] == 0), (
        'Nonzero spin-flip hopping: the spin-block solver requires zero Rashba coupling.'
    )
    return dict(sample=sample, xy=xy, positions=positions[::2], height=z[::2],
                edge=edge, image_x=image_x, coordination=coordination)


def bloch_hamiltonian(ribbon, reduced_k):
    s = ribbon['sample']
    values = s.hop_v*np.exp(2j*np.pi*reduced_k*ribbon['image_x'])
    half = coo_matrix((values, (s.hop_i, s.hop_j)), shape=(s.num_orb, s.num_orb)).tocsr()
    H = half + half.getH() + diags(s.orb_eng, format='csr')
    residual = H-H.getH()
    assert residual.nnz == 0 or abs(residual.data).max() < 1e-12
    return H

flat = build_ribbon(0.0)
bubble = build_ribbon(HEIGHT_NM)
np.testing.assert_allclose(flat['xy'], bubble['xy'])
np.testing.assert_array_equal(flat['edge'], bubble['edge'])
fig, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
for ax, title, ribbon in zip(axes, ['Flat ribbon', 'Edge-overlapping bubble'], [flat, bubble]):
    im = ax.scatter(*ribbon['xy'].T, c=ribbon['height'], s=5, vmin=0, vmax=HEIGHT_NM, cmap='viridis')
    ax.axhline(EDGE_DEPTH_NM, color='orange', ls='--', label='one-edge projector boundary')
    ax.add_patch(plt.Circle((BUBBLE_X_NM, BUBBLE_Y_NM), RADIUS_NM, fill=False, color='red'))
    ax.set(xlabel='periodic x (nm)', ylabel='open y (nm)', title=title,
           xlim=(0, L), ylim=(-0.3, WIDTH+0.3), aspect='equal')
    ax.legend(loc='upper right', fontsize=8)
fig.colorbar(im, ax=axes.tolist(), label='mid-surface height (nm)')
fig.savefig(out/'geometry.png')
plt.show()


## Spectral definitions and numerical method

Write a supercell eigenvector in the cell-periodic gauge as $u_{nq}(r,\alpha,s)$, where $r=0,\ldots,N_x-1$ labels primitive ribbon repeats, $\alpha$ labels sites across the width, and $q=k_{\rm SC}L/(2\pi)$. Its primitive momentum amplitudes are

$$\widetilde u_{nq,m}(\alpha,s)=\frac{1}{\sqrt{N_x}}\sum_r e^{-2\pi i(q+m)r/N_x}u_{nq}(r,\alpha,s),\qquad \kappa_m=\frac{q+m}{N_x}\pmod{1}.$$

The weights are $W_{nq,m}=\sum_{\alpha,s}|\widetilde u_{nq,m}(\alpha,s)|^2$ and $W^{\rm edge}_{nq,m}=\sum_{\alpha\in\mathrm{lower},s}|\widetilde u_{nq,m}(\alpha,s)|^2$. The edge projection is applied **inside** this coherent momentum projection, not by multiplying the total unfolded weight by a state's total edge weight.

$$A(\kappa_m,E)=\frac{1}{N_{\rm orb}^{\rm primitive}}\sum_n W_{nq,m}g_\eta(E-\epsilon_{nq}),\qquad A_{\rm edge}(\kappa_m,E)=\frac{1}{N_{\rm edge,orb}^{\rm primitive}}\sum_n W^{\rm edge}_{nq,m}g_\eta(E-\epsilon_{nq}).$$

The Fourier transform uses reference lattice indices, so bubble displacements do not change the momentum basis. The phase containing $q$ is essential for the boundary-only Bloch gauge used above. Parseval checks enforce $\sum_mW=1$ and $\sum_m W^{\rm edge}=\langle P_{\rm lower}\rangle$ for every state. Averaging the unfolded spectra over the primitive zone reproduces the supercell DOS and lower-edge LDOS. The original folded traces are retained in the saved arrays as `A` and `Aedge`; the main plots use `A_unfolded` and `Aedge_unfolded`.

Here $g_\eta$ is a normalized Gaussian with standard deviation 5 meV. Both spectral functions are densities **per selected spin orbital** (eV$^{-1}$). The saved spatial LDOS is instead **spin-summed per physical atom**:

$$\rho_i(E)=\frac{1}{N_k}\sum_{k,n,s}|u_{nk}(i,s)|^2g_\eta(E-\epsilon_{nk}).$$

We save LDOS maps at −50, 0 and +50 meV, LDOS integrated over [−80,+80] meV, and full energy-dependent LDOS for every atom. This interval lies inside the flat **bulk** gap; the ribbon itself has edge states, so it is not assumed gapped.

Sparse shift-invert diagonalization solves each conserved-spin block separately. The requested eigenpair count increases until the returned spectrum extends beyond both sides of the analysis window plus eight Gaussian standard deviations. A fixed cutoff then selects all retained states; omitted Gaussian tails are negligible in the plotted window. The solver raises an error if coverage, residual or orthogonality checks fail. This avoids silently dropping in-gap states. Increase `NK` and reduce broadening to assess resolution; `HB_QUICK=1` is only a coarse sampling check. These calculations can take appreciable time, but never run automatically on opening the notebook.


In [ ]:
def window_eigensystem(H):
    count = min(INITIAL_EIGENPAIRS, H.shape[0]-2)
    maximum = min(MAX_EIGENPAIRS, H.shape[0]-2)
    while True:
        # Tiny offset avoids a singular factorization at an exact zero mode.
        vals, vecs = eigsh(H, k=count, sigma=1e-9, which='LM', tol=1e-10,
                           maxiter=10000, v0=np.random.default_rng(707).normal(size=H.shape[0]))
        order = np.argsort(vals)
        vals, vecs = vals[order], vecs[:, order]
        if vals[0] < -EIGEN_CUTOFF and vals[-1] > EIGEN_CUTOFF:
            break
        if count >= maximum:
            raise RuntimeError('Eigenwindow not covered: increase MAX_EIGENPAIRS.')
        count = min(maximum, int(np.ceil(count*1.5)))
    keep = abs(vals) <= EIGEN_CUTOFF
    vals, vecs = vals[keep], vecs[:, keep]
    assert np.max(np.linalg.norm(H@vecs-vecs*vals, axis=0)) < 1e-7
    # ARPACK can return nonorthogonal vectors within exact degeneracies.
    # Rayleigh-Ritz in the retained invariant subspace restores an orthonormal basis.
    Q, _ = np.linalg.qr(vecs)
    projected = Q.conj().T @ (H@Q)
    vals, rotation = np.linalg.eigh((projected+projected.conj().T)/2)
    vecs = Q@rotation
    assert np.max(abs(vecs.conj().T@vecs-np.eye(len(vals)))) < 1e-8
    assert np.max(np.linalg.norm(H@vecs-vecs*vals, axis=0)) < 1e-7
    return vals, vecs


def gaussian(energies, eigenvalues):
    d = (np.asarray(energies)[:, None]-eigenvalues[None, :])/BROADENING_EV
    return np.exp(-0.5*d*d)/(np.sqrt(2*np.pi)*BROADENING_EV)


def unfolded_weights(vecs, reduced_k, edge):
    """Coherent primitive-momentum weights for one conserved-spin block."""
    transverse_sites = 4*ny
    amplitudes = vecs.reshape(nx, transverse_sites, vecs.shape[1])
    edge_rows = edge.reshape(nx, transverse_sites)
    assert np.all(edge_rows == edge_rows[0]), 'Edge projector must repeat along x.'
    twist = np.exp(-2j*np.pi*reduced_k*np.arange(nx)/nx)
    fourier = np.fft.fft(amplitudes*twist[:, None, None], axis=0, norm='ortho')
    intensity = abs(fourier)**2
    total = intensity.sum(axis=1)
    lower = intensity[:, edge_rows[0], :].sum(axis=1)
    np.testing.assert_allclose(total.sum(axis=0), np.sum(abs(vecs)**2, axis=0), rtol=1e-10, atol=1e-12)
    np.testing.assert_allclose(lower.sum(axis=0), np.sum(abs(vecs[edge])**2, axis=0), rtol=1e-10, atol=1e-12)
    return total, lower


def calculate(ribbon, name):
    start = perf_counter()
    nsite = len(ribbon['xy'])
    edge = ribbon['edge']
    A = np.zeros((len(ENERGY), NK))
    Aedge = np.zeros_like(A)
    A_unfolded = np.zeros((len(ENERGY), NK*nx))
    Aedge_unfolded = np.zeros_like(A_unfolded)
    ldos = np.zeros((nsite, len(ENERGY)))
    maps = np.zeros((nsite, len(MAP_ENERGIES)))
    gap_weight = np.zeros(nsite)
    eigenvalues, edge_weights, state_k, state_spin = [], [], [], []
    for ik, reduced_k in enumerate(q):
        H = bloch_hamiltonian(ribbon, reduced_k)
        for spin in (0, 1):
            vals, vecs = window_eigensystem(H[spin::2, spin::2].tocsr())
            weight = abs(vecs)**2
            ew = weight[edge].sum(axis=0)
            broad = gaussian(ENERGY, vals)
            momentum_weight, edge_momentum_weight = unfolded_weights(vecs, reduced_k, edge)
            columns = slice(ik*nx, (ik+1)*nx)
            A_unfolded[:, columns] += broad@momentum_weight.T/(2*nsite/nx)
            Aedge_unfolded[:, columns] += broad@edge_momentum_weight.T/(2*edge.sum()/nx)
            A[:, ik] += broad.sum(axis=1)/(2*nsite)
            Aedge[:, ik] += broad@ew/(2*edge.sum())
            ldos += (weight@broad.T)/NK
            maps += (weight@gaussian(MAP_ENERGIES, vals).T)/NK
            integrated = ndtr((GAP_WINDOW[1]-vals)/BROADENING_EV)-ndtr((GAP_WINDOW[0]-vals)/BROADENING_EV)
            gap_weight += weight@integrated/NK
            eigenvalues.extend(vals)
            edge_weights.extend(ew)
            state_k.extend([ik]*len(vals))
            state_spin.extend([spin]*len(vals))
        if ik % max(1, NK//10) == 0 or ik == NK-1:
            print(f'{name}: k {ik+1}/{NK}; elapsed {(perf_counter()-start)/60:.1f} min', flush=True)
    # Independent bookkeeping checks connect local and projected spectral weights.
    np.testing.assert_allclose(ldos.mean(axis=0)/2, A.mean(axis=1), rtol=1e-9, atol=1e-12)
    np.testing.assert_allclose(ldos[edge].mean(axis=0)/2, Aedge.mean(axis=1), rtol=1e-9, atol=1e-12)
    # Reorder the interlaced q,m samples into ascending primitive momentum.
    A_unfolded = A_unfolded[:, unfold_order]
    Aedge_unfolded = Aedge_unfolded[:, unfold_order]
    np.testing.assert_allclose(A_unfolded.mean(axis=1), A.mean(axis=1), rtol=1e-9, atol=1e-12)
    np.testing.assert_allclose(Aedge_unfolded.mean(axis=1), Aedge.mean(axis=1), rtol=1e-9, atol=1e-12)
    result = dict(A=A, Aedge=Aedge, A_unfolded=A_unfolded, Aedge_unfolded=Aedge_unfolded, ldos=ldos, maps=maps, gap_weight=gap_weight,
                  eigenvalues=np.asarray(eigenvalues), edge_weights=np.asarray(edge_weights),
                  state_k=np.asarray(state_k), state_spin=np.asarray(state_spin))
    for array in result.values():
        assert np.all(np.isfinite(array))
    np.savez_compressed(out/f'{name}.npz', **result, energy=ENERGY, reduced_k=q, k_nm=k_nm,
                        primitive_reduced_k=primitive_q, primitive_k_nm=primitive_k_nm,
                        map_energies=MAP_ENERGIES, gap_window=GAP_WINDOW, xy=ribbon['xy'],
                        positions=ribbon['positions'], height=ribbon['height'], edge_mask=edge)
    return result

metadata = dict(parameters=asdict(p), dimensions=[nx, ny, 1], pbc=[True, False, False],
                actual_length_nm=L, actual_atomic_width_nm=WIDTH, bubble_center_nm=[BUBBLE_X_NM, BUBBLE_Y_NM],
                shape='gaussian', radius_convention='R = 3 sigma', edge_depth_nm=EDGE_DEPTH_NM,
                spectral_energy_window_ev=[float(ENERGY[0]), float(ENERGY[-1])],
                nk=NK, gaussian_sigma_ev=BROADENING_EV, eigen_cutoff_ev=float(EIGEN_CUTOFF),
                gap_window_ev=GAP_WINDOW.tolist(), map_energies_ev=MAP_ENERGIES.tolist(),
                spectral_normalization='per selected spin orbital', ldos_normalization='spin sum per atom',
                momentum_convention='unfolded primitive ribbon BZ; kappa=ka/(2pi); folded traces also saved',
                primitive_lattice_constant_nm=p.a, unfolded_momentum_points=NK*nx, quick=QUICK)
(out/'parameters.json').write_text(json.dumps(metadata, indent=2)+'\n')


In [ ]:
# First: no bubble.
flat_result = calculate(flat, 'flat')


In [ ]:
# Second: the Gaussian bubble specified by HEIGHT_NM and RADIUS_NM.
tag = f'edge_gaussian_h{HEIGHT_NM:g}_R{RADIUS_NM:g}'.replace('.', 'p')
bubble_result = calculate(bubble, tag)


## Primitive-zone full-ribbon and one-edge spectral functions

Both spectra display the equivalent positive-momentum primitive zone $ka/(2\pi)\in[0,1]$, starting at $\Gamma$ on the left and ending at the next $\Gamma$ on the right. K and K′ are at $1/3$ and $2/3$. Negative momenta are shifted by one reciprocal lattice vector, and spectral columns are reordered with them; no states are discarded. Saved arrays retain their centred momentum convention. Flat-ribbon states recover primitive momentum selection; bubble scattering redistributes their momentum weights. Both columns keep the **−0.30 to +0.30 eV** energy window controlled above. Colour limits are shared between flat and bubble cases within each column.

There are `NK * nx` distinct primitive momenta assembled from the `NK` supercell calculations. Periodic copies of the first/last columns are added only for plotting the seam, never for averaging. The 1 nm lower-edge strip is unchanged. If `flat_result` and `bubble_result` already contain the unfolded spectra, only the plotting cell needs to be rerun for this positive-momentum display.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True, constrained_layout=True)
# Display the equivalent positive-momentum zone without recalculating spectra.
positive_q = np.mod(primitive_q, 1.0)
positive_order = np.argsort(positive_q)
positive_q = positive_q[positive_order]
k_plot = np.r_[positive_q[-1]-1, positive_q, positive_q[0]+1]
for col, (key, title) in enumerate([('A_unfolded', 'Full ribbon'), ('Aedge_unfolded', 'Lower zigzag edge only')]):
    vmax = max(flat_result[key].max(), bubble_result[key].max())
    for row, (result, name) in enumerate([(flat_result, 'Flat'), (bubble_result, 'Bubble')]):
        ax = axes[row, col]
        positive_spectrum = result[key][:, positive_order]
        z = np.column_stack((positive_spectrum[:, -1], positive_spectrum, positive_spectrum[:, 0]))
        im = ax.pcolormesh(k_plot, ENERGY, z, shading='auto', cmap='magma', vmin=0, vmax=vmax, rasterized=True)
        for bound in GAP_WINDOW:
            ax.axhline(bound, color='cyan', lw=0.6, ls=':')
        for valley in (1/3, 2/3):
            ax.axvline(valley, color='white', lw=0.7, ls='--', alpha=0.7)
        ax.set(title=f'{name}: {title}', xlabel=r'$ka/(2\pi)$', ylabel='Energy (eV)',
               xlim=(0, 1), ylim=(ENERGY[0], ENERGY[-1]))
        ax.set_xticks([0, 1/3, 0.5, 2/3, 1],
                      [r'$\Gamma$', r'$K\;(1/3)$', r'$1/2$', r"$K'\;(2/3)$", r'$\Gamma$'])
    fig.colorbar(im, ax=axes[:, col].tolist(), label='Unfolded density per selected spin orbital (1/eV)')
fig.savefig(out/'unfolded_spectral_functions.png')
plt.show()


## Spatial LDOS inside the gap

Each energy uses a shared colour scale across the two geometries. The bottom comparison integrates the broadened LDOS over the stated in-gap interval; its units are states per atom. The difference map shows bubble minus flat on the same sites. All spatial plots use undeformed in-plane coordinates to compare corresponding atoms.

Each dot is a physical atom, with LDOS summed over both spins and averaged uniformly over the supercell Bloch momenta. The energy-resolved maps use normalized Gaussian broadening of 5 meV standard deviation. They show density at the selected energy, not the probability density of a single eigenstate. Positions are the reference in-plane coordinates.

Positive maps now use logarithmic colour normalization over four decades below the shared maximum, controlled by `LDOS_COLOR_DECADES` in the plotting cell. Values below that display floor (including zero) take the darkest colour; the numerical arrays are unchanged. Flat and bubble maps share limits at each energy, and the two integrated maps share limits with each other. Colourbar labels retain physical units. The signed bubble-minus-flat map uses symmetric logarithmic normalization, with a small linear interval around zero controlled by `DIFFERENCE_LINEAR_FRACTION`. Only this plotting cell needs to be rerun.


In [ ]:
from matplotlib.colors import LogNorm, SymLogNorm

# Plot controls only: no spectral recalculation required.
LDOS_COLOR_DECADES = 4  # display down to 10**(-4) of each shared maximum
DIFFERENCE_LINEAR_FRACTION = 1e-3  # linear region around zero for signed differences
assert LDOS_COLOR_DECADES > 0
assert 0 < DIFFERENCE_LINEAR_FRACTION < 1

def ldos_color_norm(*arrays):
    peak = max(float(np.max(a)) for a in arrays)
    peak = max(peak, np.finfo(float).tiny * 10**LDOS_COLOR_DECADES)
    return LogNorm(vmin=peak * 10**(-LDOS_COLOR_DECADES), vmax=peak, clip=True)

fig, axes = plt.subplots(2, len(MAP_ENERGIES), figsize=(16, 7), constrained_layout=True)
for col, value in enumerate(MAP_ENERGIES):
    norm = ldos_color_norm(flat_result['maps'][:, col], bubble_result['maps'][:, col])
    for row, (result, name) in enumerate([(flat_result, 'Flat'), (bubble_result, 'Bubble')]):
        ax = axes[row, col]
        im = ax.scatter(*flat['xy'].T, c=result['maps'][:, col], s=7, cmap='magma', norm=norm)
        ax.set(title=f'{name}, E = {value*1000:g} meV', xlabel='x (nm)', ylabel='y (nm)', aspect='equal')
        ax.axhline(EDGE_DEPTH_NM, color='cyan', lw=0.5, ls=':')
    fig.colorbar(im, ax=axes[:, col].tolist(), label='Spin-summed LDOS (1/eV per atom; log colour)', shrink=0.8)
fig.savefig(out/'gap_ldos_maps.png')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(11, 10), constrained_layout=True)
norm = ldos_color_norm(flat_result['gap_weight'], bubble_result['gap_weight'])
for ax, result, name in zip(axes[:2], [flat_result, bubble_result], ['Flat', 'Bubble']):
    im = ax.scatter(*flat['xy'].T, c=result['gap_weight'], s=7, cmap='magma', norm=norm)
    ax.set(title=f'{name}: integrated in-gap LDOS', xlabel='x (nm)', ylabel='y (nm)', aspect='equal')
fig.colorbar(im, ax=axes[:2].tolist(), label=f'States per atom in [{1000*GAP_WINDOW[0]:g}, {1000*GAP_WINDOW[1]:g}] meV (log colour)')
difference = bubble_result['gap_weight']-flat_result['gap_weight']
limit = max(abs(difference).max(), 1e-15)
difference_norm = SymLogNorm(linthresh=limit*DIFFERENCE_LINEAR_FRACTION,
                             vmin=-limit, vmax=limit, base=10)
im = axes[2].scatter(*flat['xy'].T, c=difference, s=7, cmap='RdBu_r', norm=difference_norm)
axes[2].set(title='Bubble minus flat', xlabel='x (nm)', ylabel='y (nm)', aspect='equal')
fig.colorbar(im, ax=axes[2], label='Change in states per atom (symmetric log colour)')
fig.savefig(out/'integrated_gap_ldos.png')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for result, name in [(flat_result, 'Flat'), (bubble_result, 'Bubble')]:
    axes[0].plot(ENERGY, result['A'].mean(axis=1), label=name)
    axes[1].plot(ENERGY, result['Aedge'].mean(axis=1), label=name)
for ax, title in zip(axes, ['Full-ribbon mean DOS', 'Lower-edge mean LDOS']):
    ax.axvspan(*GAP_WINDOW, color='grey', alpha=0.15)
    ax.set(title=title, xlabel='Energy (eV)', ylabel='Density per spin orbital (1/eV)')
    ax.legend()
fig.savefig(out/'bz_averaged_spectra.png')
plt.show()
print(f'Arrays, parameters and figures saved to {out.resolve()}')
